## Analyze the dataset distribution among classes.

Evaluate feature importance to compute the most important feature and remove the non important ones, then create a binary dataset so that we can work with a GAN. Actually here we may think of using another apporach, consisering multiple classes.

The inital dataset is composed by the following classes:<br>
NORMAL           17816<br>
c_rd_na_1_DoS     4974<br>
c_rp_na_1_DoS     4970<br>
c_sc_na_1_DoS     3688<br>
c_rp_na_1         2900<br>
c_rd_na_1         2720<br>
c_se_na_1_DoS     2686<br>
c_ci_na_1_DoS     1986<br>
c_se_na_1         1728<br>
c_sc_na_1         1482<br>
m_sp_na_1_DoS     1274<br>
c_ci_na_1         1140<br><br>

So we can drop the column normal and only uses the other classes.<br>

In [1]:
import os
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

def save_artifacts(out_dir: Path, X_train, X_val, y_train, y_val, feat_names):
    art_dir = out_dir / "artifacts"
    art_dir.mkdir(parents=True, exist_ok=True)

    # scaler
    scaler = StandardScaler()
    Xtr_scaled = scaler.fit_transform(X_train.values).astype(np.float32)
    Xva_scaled = scaler.transform(X_val.values).astype(np.float32)

    # salva feature list
    with open(art_dir / "features.json", "w", encoding="utf-8") as f:
        json.dump(list(feat_names), f, indent=2)

    # salva scaler
    with open(art_dir / "scaler.pkl", "wb") as f:
        pickle.dump(scaler, f)

    # salva CSV per GAN
    df_tr = pd.DataFrame(Xtr_scaled, columns=feat_names)
    df_tr["label"] = y_train.values.astype(np.int32)
    df_va = pd.DataFrame(Xva_scaled, columns=feat_names)
    df_va["label"] = y_val.values.astype(np.int32)
    df_tr.to_csv(out_dir / "gan_train_raw.csv", index=False)
    df_va.to_csv(out_dir / "gan_val_raw.csv", index=False)



def to_binary_label(val):
    """0=benigno, 1=maligno. Gestisce stringhe multiclass CICFlow: tutto ciò che non è BENIGN/normal è 1."""
    try:
      v = val.lower()
      if v.endswith("dos"):
        return 1
      return 0
    except AttributeError as e:
      print("Errore " + str(val))
      return 0



input_csv='merged_network_flow_leayer.csv'
out_dir="."
label_col=None
topk_plot=10
test_size=0.2
random_state=42
select_topk=7
out_dir = Path(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
label_col = "Label"

df = pd.read_csv(input_csv)
df = df.dropna(axis=1, how="all")
df = df.dropna(subset=[label_col])

y = df[label_col].apply(to_binary_label).astype(np.int32)

numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]

# Remove the features related to the identity of the node
DROP_IF_PRESENT = {
    "timestamp", "time", "date", "datetime", "ts", "flow_start", "flow_end",
    "src_ip", "dst_ip", "source_ip", "destination_ip",
    "src_port", "dst_port", "source_port", "destination_port",
    "sport", "dport", "proto", "protocol", "flow_id", "id", "uid", "src port"
}
feat_cols = [c for c in numeric_cols if c != label_col and c.lower() not in DROP_IF_PRESENT]

# Resolve missing types
X_full = df[feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(np.float32)

# split
Xtr_full, Xva_full, ytr, yva = train_test_split(
    X_full, y, test_size=test_size, random_state=random_state, stratify=y
)

# feature importance (sul full)
rf = RandomForestClassifier(
    n_estimators=300, random_state=13, n_jobs=-1, class_weight="balanced_subsample"
)
rf.fit(Xtr_full, ytr)
fi = pd.Series(rf.feature_importances_, index=X_full.columns).sort_values(ascending=False)
fi.to_csv(out_dir / "feature_importance.csv", header=["gini_importance"])

# plot top-k (per il grafico usiamo topk_plot, indipendentemente da select_topk)
plot_series = fi.head(int(topk_plot))[::-1]
plt.figure(figsize=(10, max(5, len(plot_series) * 0.3 + 2)))
plt.barh(plot_series.index, plot_series.values)
plt.xlabel("Importance")
plt.title(f"Top-{int(topk_plot)} feature importances")
plt.tight_layout()
plt.savefig(out_dir / "feature_importance_topk.png", dpi=160)
plt.close()

topk_feats = list(fi.head(select_topk).index)

# Apply the feature importance list on the dataset (both training and validation datasets)
Xtr_k = Xtr_full[topk_feats].copy()
Xva_k = Xva_full[topk_feats].copy()

# sovrascrive gli artifact GAN con la versione top-k
save_artifacts(out_dir, Xtr_k, Xva_k, ytr, yva, topk_feats)

selected_info = {
    "selected_topk": select_topk,
    "selected_features": topk_feats
}
(out_dir / "selected_topk.json").write_text(json.dumps(selected_info, indent=2), encoding="utf-8")


# meta
meta = {
    "input_csv": str(Path(input_csv).resolve()),
    "out_dir": str(out_dir.resolve()),
    "label_col": label_col,
    "n_rows": int(df.shape[0]),
    "n_features_full": int(X_full.shape[1]),
    "train_size": int(len(ytr)),
    "val_size": int(len(yva)),
    "malign_rate_train": float(np.mean(ytr)),
    "malign_rate_val": float(np.mean(yva)),
    "topk_plot": int(topk_plot),
    "select_topk": int(select_topk) if select_topk is not None else None
}
(out_dir / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

395

In [2]:
import os, time, json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import precision_recall_fscore_support

# ----------------- Utils -----------------

def make_dataloader(x: np.ndarray, y: np.ndarray, batch: int, shuffle=True):
    dataset = TensorDataset(torch.tensor(x, dtype=torch.float32),
                            torch.tensor(y, dtype=torch.long))
    return DataLoader(dataset, batch_size=batch, shuffle=shuffle)

def eval_disc_on(dl, D, device):
    accs, rfps = [], []
    D.eval()
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = D(xb)
            pred = torch.argmax(logits, dim=-1)
            accs.append((pred == yb).float().mean().item())
            prob = F.softmax(logits, dim=-1)
            rfps.append((prob[:, 2] > 0.5).float().mean().item())
    return np.mean(accs), np.mean(rfps)

def eval_supervised_metrics(dl, D, device):
    ys, yh = [], []
    D.eval()
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = D(xb)
            pred = torch.argmax(logits[:, :2], dim=-1)
            ys.append(yb.cpu().numpy())
            yh.append(pred.cpu().numpy())
    if not ys:
        return 0.0, 0.0, 0.0
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(yh)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[1], average=None, zero_division=0)
    return float(p[0]), float(r[0]), float(f[0])

# ----------------- Models -----------------

class Generator(nn.Module):
    def __init__(self, latent_dim, output_dim, hidden=(256, 128, 64), dropout=0.0):
        super().__init__()
        layers = []
        last = latent_dim
        for i, h in enumerate(hidden):
            layers.append(nn.Linear(last, h))
            layers.append(nn.LayerNorm(h))
            layers.append(nn.LeakyReLU(0.2))
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            last = h
        layers.append(nn.Linear(last, output_dim))
        self.model = nn.Sequential(*layers)
    def forward(self, z):
        return self.model(z)

class Discriminator(nn.Module):
    def __init__(self, input_dim, hidden=(256,128,64), dropout=0.1, l2=1e-4):
        super().__init__()
        layers = []
        last = input_dim
        for h in hidden:
            layers.append(nn.Linear(last,h))
            layers.append(nn.LeakyReLU(0.2))
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            last = h
        self.features = nn.Sequential(*layers)
        self.logits = nn.Linear(last, 3)
    def forward(self, x):
        feat = self.features(x)
        out = self.logits(feat)
        return out
    def extract_features(self, x):
        return self.features(x)

# ----------------- Training steps -----------------

def disc_step(D, G, opt, real_x, real_y, latent_dim, label_smooth, noise_std, class_weights, device):
    bsz = real_x.size(0)
    D.train()
    if noise_std > 0:
        real_x = real_x + torch.randn_like(real_x) * noise_std
    real_x, real_y = real_x.to(device), real_y.to(device)
    
    z = torch.randn(bsz, latent_dim, device=device)
    fake_x = G(z)
    if noise_std > 0:
        fake_x = fake_x + torch.randn_like(fake_x) * noise_std

    logits_real = D(real_x)
    logits_fake = D(fake_x)

    y_real_onehot = F.one_hot(real_y, num_classes=3).float()
    if label_smooth > 0:
        y_real_onehot = y_real_onehot * (1-label_smooth) + label_smooth/3.0

    cls_w = class_weights.to(device)[real_y]
    loss_sup = (F.cross_entropy(logits_real, real_y, reduction='none') * cls_w).mean()

    y_fake = torch.full((bsz,), 2, dtype=torch.long, device=device)
    loss_unsup = F.cross_entropy(logits_fake, y_fake)
    
    loss = loss_sup + loss_unsup

    opt.zero_grad()
    loss.backward()
    opt.step()

    pred_real = torch.argmax(logits_real, dim=-1)
    acc_real = (pred_real == real_y).float().mean().item()
    rfp_real = (F.softmax(logits_real, dim=-1)[:,2] > 0.5).float().mean().item()
    return loss.item(), acc_real, rfp_real

def gen_step(D, G, F, opt, real_x, latent_dim, noise_std, fm_w, pi0, pi1, device):
    bsz = real_x.size(0)
    G.train()
    real_x = real_x.to(device)
    z = torch.randn(bsz, latent_dim, device=device)
    fake_x = G(z)
    if noise_std > 0:
        fake_x = fake_x + torch.randn_like(fake_x) * noise_std
    logits_fake = D(fake_x)

    target = torch.cat([
        torch.full((bsz,1), pi0, device=device),
        torch.full((bsz,1), pi1, device=device),
        torch.zeros((bsz,1), device=device)
    ], dim=1)
    g_loss = F.cross_entropy(logits_fake, target.argmax(dim=1))

    if fm_w > 0:
        F_real = F.extract_features(real_x)
        F_fake = F.extract_features(fake_x)
        fm_loss = ((F_real.mean(0)-F_fake.mean(0))**2).mean()
        g_loss += fm_w * fm_loss

    opt.zero_grad()
    g_loss.backward()
    opt.step()

    fooled = (torch.argmax(F.softmax(logits_fake, dim=-1), dim=-1) != 2).float().mean().item()
    return g_loss.item(), fooled

# ----------------- Main -----------------

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    # --- Config ---
    args = {
        "epochs": 50,
        "batch_size": 128,
        "latent_dim": 64,
        "lr_g": 1e-4,
        "lr_d": 2e-4,
        "g_steps": 1,
        "d_steps": 3,
        "label_smooth": 0.1,
        "noise_std": 0.005,
        "fm_w": 0.5,
        "class_weight_0": 1.0,
        "class_weight_1": 1.0,
        "data_root": "."
    }
    args = type("Args", (), args)()

    DATA_ROOT = Path(args.data_root).resolve()
    ART = DATA_ROOT / "artifacts"
    TR_RAW = DATA_ROOT / "gan_train_raw.csv"
    VA_RAW = DATA_ROOT / "gan_val_raw.csv"

    df_tr = pd.read_csv(TR_RAW)
    df_va = pd.read_csv(VA_RAW)
    features = [c for c in df_tr.columns if c != "label"]

    Xtr, ytr = df_tr[features].to_numpy(dtype=np.float32), df_tr["label"].to_numpy(dtype=np.int64)
    Xva, yva = df_va[features].to_numpy(dtype=np.float32), df_va["label"].to_numpy(dtype=np.int64)

    ds_train = make_dataloader(Xtr, ytr, batch=args.batch_size, shuffle=True)
    ds_val   = make_dataloader(Xva, yva, batch=args.batch_size, shuffle=False)

    input_dim = Xtr.shape[1]

    # --- Modelli ---
    G = Generator(args.latent_dim, input_dim).to(device)
    D = Discriminator(input_dim).to(device)
    F = D  # feature extractor = D senza logits, useremo extract_features

    # --- Optimizers ---
    gen_opt = torch.optim.Adam(G.parameters(), lr=args.lr_g, betas=(0.5, 0.999))
    disc_opt = torch.optim.Adam(D.parameters(), lr=args.lr_d, betas=(0.5, 0.999))
    class_weights = torch.tensor([args.class_weight_0, args.class_weight_1, 1.0], dtype=torch.float32)

    # --- Training loop ---
    for epoch in range(1, args.epochs+1):
        dl_acc, gl_acc, fr_acc, a_tr_acc, rfp_tr_acc = [], [], [], [], []
        ys_tr, yh_tr = [], []

        for xb, yb in ds_train:
            xb, yb = xb.to(device), yb.to(device)
            for _ in range(args.d_steps):
                d_loss, acc_real, rfp_real = disc_step(D, G, disc_opt, xb, yb, args.latent_dim,
                                                      args.label_smooth, args.noise_std, class_weights, device)
            dl_acc.append(d_loss)
            a_tr_acc.append(acc_real)
            rfp_tr_acc.append(rfp_real)

            logits = D(xb)
            pred = torch.argmax(F.softmax(logits, dim=-1)[:, :2], dim=-1)
            ys_tr.append(yb.cpu().numpy())
            yh_tr.append(pred.cpu().numpy())

            g_losses, fools = [], []
            for _ in range(args.g_steps):
                g_loss, fooled = gen_step(D, G, F, gen_opt, xb, args.latent_dim,
                                          args.noise_std, args.fm_w, 0.7, 0.3, device)
                g_losses.append(g_loss)
                fools.append(fooled)
            gl_acc.append(np.mean(g_losses))
            fr_acc.append(np.mean(fools))

        # Metriche supervised train
        y_true_tr = np.concatenate(ys_tr)
        y_pred_tr = np.concatenate(yh_tr)
        p_tr, r_tr, f_tr, _ = precision_recall_fscore_support(y_true_tr, y_pred_tr, labels=[1], average=None, zero_division=0)
        prec1_tr, rec1_tr, f1_1_tr = float(p_tr[0]), float(r_tr[0]), float(f_tr[0])

        acc_val, rfp_val = eval_disc_on(ds_val, D, device)
        prec1_val, rec1_val, f1_1_val = eval_supervised_metrics(ds_val, D, device)

        print(f"[E{epoch}] D_loss={np.mean(dl_acc):.4f}, G_loss={np.mean(gl_acc):.4f}, "
              f"ACC(tr)={np.mean(a_tr_acc):.3f}, ACC(val)={acc_val:.3f}, "
              f"rFP(tr)={np.mean(rfp_tr_acc):.3f}, rFP(val)={rfp_val:.3f}, "
              f"FR(tr)={np.mean(fr_acc):.3f}, F1_1(tr)={f1_1_tr:.3f}, F1_1(val)={f1_1_val:.3f}")

if __name__ == "__main__":
    main()


Device: cuda


AttributeError: 'Discriminator' object has no attribute 'softmax'